# Example 12.7 / 2: Write Complementarity with complements

Companion notebook for **12.7 Bilevel Optimization** in the AMPL lab Documentation, section "Write Complementarity with complements". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Write Complementarity with complements

The same KKT system can use `complements` instead of binaries. HiGHS has no native complementarity support, so AMPL's MP interface converts each pair with big-M constraints; that needs finite bounds on the multipliers, or the solver option `cvt:bigM`. The run solves once without bounds, then with `cvt:bigM=100`, and finally drops the KKT rows to show what happens when the follower is treated as ordinary constraints.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var X >= 0 <= 6;                        # leader decision
var Y >= 0;                             # follower response
var Mu1 >= 0;                           # multiplier of FollowerLimit1
var Mu2 >= 0;                           # multiplier of FollowerLimit2
var Nu >= 0;                            # multiplier of Y >= 0

minimize LeaderCost: X + 2 * Y;

subject to FollowerLimit1: Y <= 1 + X;
subject to FollowerLimit2: Y <= 6 - X;
subject to Stationarity: Mu1 + Mu2 - Nu = 1;
subject to Pair1: Mu1 >= 0 complements 1 + X - Y >= 0;
subject to Pair2: Mu2 >= 0 complements 6 - X - Y >= 0;
subject to PairY: Nu >= 0 complements Y >= 0;

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)

ampl.solve(solver="highs", highs_options="cvt:bigM=100", verbose=False)
print(ampl.solve_result, ampl.get_value("X"), ampl.get_value("Y"),
      ampl.obj["LeaderCost"].value())

# Follower treated as ordinary constraints: no KKT rows.
ampl.eval("problem Naive: X, Y, LeaderCost, FollowerLimit1, FollowerLimit2;")
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.get_value("X"), ampl.get_value("Y"),
      ampl.obj["LeaderCost"].value())

failure
solved 0 1 2.0
solved 0 0 0.0


### Pure AMPL: model.run

The same in pure AMPL, with the same `model.mod`:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
option solver highs;
solve;
option highs_options 'cvt:bigM=100';
solve;
display X, Y, LeaderCost;
problem Naive: X, Y, LeaderCost, FollowerLimit1, FollowerLimit2;
solve;
display X, Y, LeaderCost;
""")
except Exception:
    pass  # AMPL has printed the error above

[WARNING] 
	Nonsquare complementarity system:
		4 complementarities including 1 equations
		5 variables


HiGHS 1.14.0: Set bounds on variables
participating in logical expressions,
or use option cvt:bigM (with caution).
See more: mp.ampl.com/modeling-numerics.html
HiGHS 1.14.0:   cvt:bigM = 100
HiGHS 1.14.0: optimal solution; objective 2
5 simplex iterations
1 branching nodes
X = 0
Y = 1
LeaderCost = 2

HiGHS 1.14.0:   cvt:bigM = 100
HiGHS 1.14.0: optimal solution; objective 0
0 simplex iterations
0 barrier iterations
X = 0
Y = 0
LeaderCost = 0

